<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers faiss-cpu

Mount The Drive

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Packages

In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Sentence Tranformer

In [4]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


Read the Data

In [6]:
chunks = []

#with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
#with open("/content/drive/MyDrive/AI_capstone_training_data/all_records_final_formatted_3.json", encoding="utf-8") as f:
with open("/content/drive/MyDrive/AI_capstone_training_data/positive_records_singular.json", encoding="utf-8") as f:
    records = json.load(f)
    #for record in records["All_Records"]:
    for record in records["Positive_Records"]:
        chunks.append(record)


Create Pragraphs With Metadata

In [7]:
paragraphs = [
    chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
All financial statement schedules have been omitted, since the required information is not applicable or is not present in amounts sufficient to require submission of the schedule, or because the informatio
Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
All financial statement schedules have been omitted, since the required information is not applicable or is not present in amounts sufficient to require submission of the schedule, or because the informatio
Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
All financial statement schedules have been omitted, since the required information is not applicable or is not present in amounts sufficient to require submission of the schedule, or because the informatio


Create Embeddings

In [8]:


embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)
print(embeddings.shape)

Batches:   0%|          | 0/501 [00:00<?, ?it/s]

(64070, 384)


In [9]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS vectors:", index.ntotal)
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))


FAISS vectors: 64070
Number of chunks: 64070
Number of embeddings: 64070


Write Indexes

In [10]:


faiss.write_index(
    index,
    "financial_reports.index"
)

In [11]:
#Check the file

import os

size_mb = os.path.getsize(
    "financial_reports.index"
) / (1024 * 1024)

print(f"Index size: {size_mb:.2f} MB")

Index size: 93.85 MB


In [12]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")
    """tks = chunk["metadata"].split(" ")
    ticker = (tks[1]).split("-")[1]
    section = tks[len(tks)-1].split("-")[1]
    date = tks[len(tks)-2].split("-")[1]
    company = tks[2].split("-")[1]
    metadata.append({
        "ticker": ticker,
        "section": section,
        "year": date,
        "reference": chunk["reference"]
    })"""
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "reference": chunk["reference"]
    })

In [13]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [14]:
import os

print(
    "File size:",
    os.path.getsize("financial_reports.index"),
    "bytes"
)

File size: 98411565 bytes


Read Indexes

In [15]:
"""index = faiss.read_index(
    "/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever/financial_reports.index"
)
with open("/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever/financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)"""
#index = faiss.IndexFlatIP(384)
index = faiss.read_index(
    "financial_reports.index"
)
with open("financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)
#index = faiss.IndexFlatIP(384)
#index.add(embeddings)

Search For Top 50 Answers Based On Question Encoding

In [16]:
question = "What is Apple's revenue"

"""query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True#,
    #normalize_embeddings=True
)

print(query_embedding.shape)
print(query_embedding[0][:10])"""

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

scores, indices = index.search(
    query_embedding,
    50
)

print("Unique indices:", len(set(indices[0])))
print(indices[0])

Unique indices: 50
[1565 1564 1567 1566  654  653  652  651  650  649  648  142  141  140
  139  138  137  136  431  430  429  428  427  535  534  533  532   23
   22   21   20 1583 1582  897  896  895  894  893  892  891  890  889
  888  385  384  383  382  381  380  379]


In [17]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

[[0.6725098  0.6725098  0.6603533  0.6603533  0.65736806 0.65736806
  0.65736806 0.65736806 0.65736806 0.65736806 0.65736806 0.65736806
  0.65736806 0.65736806 0.65736806 0.65736806 0.65736806 0.65736806
  0.64648163 0.64648163 0.64648163 0.64648163 0.64648163 0.6451341
  0.6451341  0.6451341  0.6451341  0.6451341  0.6451341  0.6451341
  0.6451341  0.6418222  0.6418222  0.61445874 0.61445874 0.61445874
  0.61445874 0.61445874 0.61445874 0.61445874 0.61445874 0.61445874
  0.61445874 0.61445874 0.61445874 0.61445874 0.61445874 0.61445874
  0.61445874 0.61445874]]
[[1565 1564 1567 1566  654  653  652  651  650  649  648  142  141  140
   139  138  137  136  431  430  429  428  427  535  534  533  532   23
    22   21   20 1583 1582  897  896  895  894  893  892  891  890  889
   888  385  384  383  382  381  380  379]]


In [18]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    #file_output = open("retrieved_results1.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    """dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]"""
    dict["reference"] = chunk["reference"]
    dict["chunk_id"] = chunk["chunk_id"]
    print(dict["reference"])
    print(dict["chunk_id"])
    #json.dump(dict, file_output)





Section:7,Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
CompanyName:Apple
Ticker:AAPL
Year:2025
Apple Inc. | 2025 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions): ##TABLE_START 2025 Change 2024 Change 2023 iPhone $ 209,586 4 % $ 201,183 &#8212; % $ 200,583 Mac 33,708 12 % 29,984 2 % 29,357 iPad 28,023 5 % 26,694 (6) % 28,300 Wearables, Home and Accessories 35,686 (4) % 37,005 (7) % 39,845 Services (1)

AAPL-7-2025-17
Section:7,Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
CompanyName:Apple
Ticker:AAPL
Year:2025
Apple Inc. | 2025 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions): ##TABLE_START 2025 Change 2024 Change 2023 iPhone $ 209,586 4 % $ 201,183 &#8212; % $ 200,583 Mac 33,708 12 % 29,984 2 % 29,357 iPad

In [14]:
for rank, (score, ids) in enumerate(zip(scores[0], indices[0])):
    print(
        f"{rank+1}: idx={ids}, score={score:.4f}"
    )

1: idx=123221, score=0.6725
2: idx=112761, score=0.6725
3: idx=17441, score=0.6725
4: idx=120586, score=0.6604
5: idx=115211, score=0.6604
6: idx=112280, score=0.6604
7: idx=105758, score=0.6604
8: idx=85464, score=0.6604
9: idx=81267, score=0.6604
10: idx=69723, score=0.6604
11: idx=64760, score=0.6604
12: idx=56888, score=0.6604
13: idx=54775, score=0.6604
14: idx=52623, score=0.6604
15: idx=47444, score=0.6604
16: idx=44529, score=0.6604
17: idx=35157, score=0.6604
18: idx=30476, score=0.6604
19: idx=14520, score=0.6604
20: idx=14088, score=0.6604
21: idx=115534, score=0.6574
22: idx=98124, score=0.6574
23: idx=96856, score=0.6574
24: idx=93804, score=0.6574
25: idx=93529, score=0.6574
26: idx=87513, score=0.6574
27: idx=87001, score=0.6574
28: idx=83207, score=0.6574
29: idx=72703, score=0.6574
30: idx=71842, score=0.6574
31: idx=66187, score=0.6574
32: idx=65575, score=0.6574
33: idx=47951, score=0.6574
34: idx=47359, score=0.6574
35: idx=46272, score=0.6574
36: idx=43954, score=0

Copy Results To Drive

In [16]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

All .json files copied successfully!
